In [6]:
#  The Solution: Build an AI Assistant
# The university deploys an AI model that can pre-screen essays and alert professors about suspicious content.
# But here’s the challenge…
# ❗ Academic plagiarism datasets are usually small and domain-specific.
# Training a deep NLP model from scratch would require:
# - Millions of labeled essays
# - Huge compute resources
# - Months of training
# Not practical.

# ⭐ Enter Transfer Learning (Your Code)
# Instead of starting from zero, engineers use:
# 👉 GPT-style language models trained on massive text corpora (books, articles, Wikipedia).
# Although these corpora contain general language, the early layers learn universal text features, like:
# - ✅ Grammar patterns
# - ✅ Semantic meaning
# - ✅ Sentence structures
# - ✅ Contextual relationships
# These features are also present in student essays.
# By fine-tuning the model on a smaller plagiarism dataset, the AI can quickly learn to distinguish:
# - Original vs. copied content
# - Paraphrased vs. genuinely written text
# - Citation misuse vs. proper referencing

# ⚡ Just like ResNet50 helps radiologists with X-rays, and BERT helps fraud analysts with transactions, GPT-style models help
# educators with essays — all leveraging transfer learning to solve problems where data is scarce but accuracy is critical.


In [7]:
# pip install transformers torch datasets pandas scikit-learn

In [8]:
import pandas as pd

data = {
    "essay": [
        "Artificial intelligence is transforming modern education by enabling personalized learning.",
        "Artificial intelligence is transforming modern education by enabling personalized learning.",  # copied
        "Climate change affects ecosystems and biodiversity worldwide.",
        "Global warming impacts wildlife habitats and ecological balance."
    ],
    "label": [0, 1, 0, 0]  # 0 = original, 1 = plagiarized
}

df = pd.DataFrame(data)
print(df)

                                               essay  label
0  Artificial intelligence is transforming modern...      0
1  Artificial intelligence is transforming modern...      1
2  Climate change affects ecosystems and biodiver...      0
3  Global warming impacts wildlife habitats and e...      0


In [9]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: distilgpt2
Key                                        | Status     | 
-------------------------------------------+------------+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED | 
score.weight                               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

In [11]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

def tokenize(example):
    return tokenizer(
        example["essay"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize)
dataset = dataset.rename_column("label", "labels")

dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [12]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./plagiarism_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=1.7355335553487141, metrics={'train_runtime': 17.3662, 'train_samples_per_second': 0.691, 'train_steps_per_second': 0.173, 'total_flos': 391959281664.0, 'train_loss': 1.7355335553487141, 'epoch': 3.0})

In [13]:
import torch

def detect_plagiarism(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    labels = {
        0: "Original Essay",
        1: "Plagiarized Content"
    }

    return labels[prediction]

In [14]:
essay = "Artificial intelligence is transforming modern education by enabling personalized learning."

result = detect_plagiarism(essay)

print("Essay:", essay)
print("Prediction:", result)

Essay: Artificial intelligence is transforming modern education by enabling personalized learning.
Prediction: Original Essay
